# baseline v4 (Unsloth + 점수 개선판, 오프라인 버전)

`(260902)_baseline_desktop5060ti_offline.ipynb`(baseline v3)를 **Unsloth** 라이브러리로 옮기고, 점수에 영향이 큰 문제들을 고친 버전입니다.
RTX 5060 Ti 16GB 에서 개발·검증했습니다. **대회 중 인터넷 다운로드·`pip install`을 하지 않습니다.**

| # | baseline v3 | v4 (이 노트북) | 왜 점수가 오르나 |
|---|---|---|---|
| 1 | Qwen2.5-VL-3B + transformers/peft | **Qwen3-VL-4B (Unsloth 4bit)** + `FastVisionModel` | 더 최신·큰 VL 모델, Unsloth 커널로 학습 가속·VRAM 절약 |
| 2 | 총 픽셀 384×384(약 0.15MP)로 고정 → 크게 축소 | **픽셀 예산 1024×1024** → 대부분(720×960)의 이미지를 **원본 크기 그대로** 사용 | 문제 대부분이 간판·가격표 글자 읽기(OCR) → 글자가 보여야 맞힘 |
| 3 | 학습 데이터 200개 | **train 전체** (약 6.2천 문항) | 학습량 30배 |
| 4 | 라벨 마스킹 없음 (질문·이미지 토큰까지 loss) | **정답 응답 토큰에만 loss** (정답 글자 + 종료 토큰, `train_on_responses_only`) | 정답 신호가 수백 개의 질문·이미지 토큰에 묻히지 않음 |
| 5 | fp16/bf16 혼용 + GradScaler | **bf16 통일**, adamw_8bit, cosine 스케줄 | 수치 안정성·메모리 |
| 6 | 검증은 loss 만 | **검증 정확도**로 주기적 평가 + **best 어댑터 자동 저장** | 과적합 전 최고점을 골라 제출 |
| 7 | `generate` + 문자열 파싱 (실패 시 "a") | **a/b/c/d 토큰 확률 직접 비교** + 배치 추론 | 파싱 실패 0, 1회 forward 로 빠름 |
| 8 | - | **보기 순서 섞기 증강 + 보기 순서 TTA** | 위치(글자) 편향 제거 |

**폴더 구조** (모델은 `downloads/models/` → 없으면 HF 캐시 순서로 찾습니다)

```md
├─ (260921)_baseline_unsloth_desktop5060ti_offline.ipynb   ← 이 노트북
├─ data/        train.csv, test.csv, dev.csv, sample_submission.csv, train/, test/, dev/
├─ downloads/models/Qwen3-VL-4B-Instruct-unsloth-bnb-4bit/  (선택: 없으면 HF 캐시 사용)
├─ downloads/models/KFC/                                    (선택: 공개 LoRA 어댑터. 없으면 HF 캐시 사용)
└─ output/
   ├─ submission.csv                          ← 제출 파일 (마지막 전체 실행 결과. QUICK_RUN 은 덮어쓰지 않음)
   ├─ submission_{RUN_NAME}_{INFER_TAG}.csv   ← 실험별 제출 파일 (RUN_NAME·INFER_TAG 는 설정값으로 자동 생성)
   ├─ probs_test_{RUN_NAME}_{INFER_TAG}.npy   ← 실험별 test 확률 (앙상블용)
   ├─ {RUN_NAME}_best/                        ← 최고점 LoRA 어댑터
   └─ {RUN_NAME}_log.txt                      ← 검증 정확도·loss 기록
```

**실행 순서** : 위에서부터 차례대로 실행. 처음이라면 `QUICK_RUN = True` 로 5분 정도 전체 흐름이 도는지 먼저 확인하세요.

### 선학습한 공개 LoRA 가중치에 연결

이 노트북을 `INIT_ADAPTER = None`(처음부터 학습)으로 실행해 얻은 LoRA 어댑터입니다. 설정 셀의 두 값으로 사용 방식을 고릅니다.

| 하고 싶은 것 | `INIT_ADAPTER` | `DO_TRAIN` | 걸리는 시간 (RTX 5060 Ti) |
|---|---|---|---|
| **이어서 파인튜닝 → 추론** (기본값) | - | `True` | 학습 2시간 20분 + 추론 2시간 |
| 그대로 **추론만** | - | `False` | 추론 2시간 (`N_TTA=1` 이면 1시간) |
| 원본 모델에서 **처음부터** 파인튜닝 | `None` | `True` | 학습 2시간 20분 + 추론 2시간 |

- 이어서 학습할 때 LoRA 구조(r=16, alpha=16, 비전 LoRA)는  설정이 자동으로 맞춰지고, LR 기본값은 5e-5 입니다.
- `train.csv` 에서 검증 500문항(`SPLIT_SEED=42`)을 뺀 나머지로 학습됐습니다. 검증 점수가 공정하도록 `SPLIT_SEED=42`, `VALID_SIZE<=500` 을 유지하세요. (다르면 에러로 막음)
- 이어서 학습해도 검증 점수가 이 보다 오르지 않으면 step 0(그대로)가 best 로 선택되어 점수가 떨어지지 않습니다.

**학습 기록 (INIT_ADAPTER=None, 검증 500문항)** : zero-shot 91.6% → 파인튜닝 94.4% (best step 600/777) → + TTA 2회 **95.2%**

| step | 0 (zero-shot) | 150 | 300 | 450 | 600 | 750 | 777 |
|---|---|---|---|---|---|---|---|
| 검증 정확도 | 91.6% | 93.6% | 93.8% | 93.8% | **94.4%** | 94.4% | 94.2% |

**소요 시간 (RTX 5060 Ti 실측)** : 모델 로드 1분 → step 0 검증 3분 → 학습 2시간 20분(검증 평가 포함) → test 추론(TTA 2회) 약 2시간 (`N_TTA=1` 이면 절반)

### 사전 준비 (대회 전, 인터넷 연결 상태에서 1회)

검증한 라이브러리 버전 : `torch 2.11.0+cu128`, `transformers 5.5.0`, `trl 0.24.0`, `peft 0.20.0`, `bitsandbytes 0.50.0`, `unsloth 2026.9.2`, `unsloth_zoo 2026.9.3`
(baseline v3 의 가상환경에는 unsloth 가 없고 transformers 버전이 달라서 그대로는 동작하지 않습니다.)

모델과 어댑터는 아래 코드로 미리 받아 둡니다. (오프라인용으로는 실행하지 마세요)
```python
from huggingface_hub import snapshot_download
snapshot_download("unsloth/Qwen3-VL-4B-Instruct-unsloth-bnb-4bit",
                  local_dir="downloads/models/Qwen3-VL-4B-Instruct-unsloth-bnb-4bit")
snapshot_download("ssafyjinhyeok/KFC", local_dir="downloads/models/KFC")
```

# 설정

이 셀만 바꿔서 실험합니다. **환경변수는 unsloth / transformers 를 import 하기 전에** 설정해야 적용됩니다.

In [ ]:
import os, time

# ── 오프라인 모드 : 반드시 unsloth / transformers import "전"에 설정 ──
os.environ["HF_HUB_OFFLINE"] = "1"               # 모델 로드 시 인터넷 접속 시도 자체를 차단
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["UNSLOTH_DISABLE_STATISTICS"] = "1"   # Unsloth 사용 통계 전송(네트워크) 끄기
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ── 경로 ──
ASSET_DIR  = "downloads"
DATA_DIR   = "data"
OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── 사전학습 모델 : 이미지를 입력받는 VL(Vision-Language) 모델이어야 합니다 ──
MODEL_NAME = "unsloth/Qwen3-VL-4B-Instruct-unsloth-bnb-4bit"     # Unsloth 가 미리 4bit 양자화한 모델
# MODEL_NAME = "unsloth/Qwen3-VL-8B-Instruct-unsloth-bnb-4bit"   # 더 크고 정확하지만 학습·추론 약 2배 느림 (사전 다운로드 필요)

def resolve_model_dir(name):
    """로컬 폴더 경로면 그대로, downloads/models/<폴더> 가 있으면 그 폴더, 없으면 HF 캐시 폴더 (모두 인터넷 접속 없음)"""
    if os.path.isdir(name):
        return name
    local = os.path.join(ASSET_DIR, "models", name.split("/")[-1])
    if os.path.isdir(local) and os.listdir(local):
        return local
    from huggingface_hub import snapshot_download
    return snapshot_download(name, local_files_only=True)   # 캐시에 없으면 에러 → 사전 다운로드 필요

MODEL_DIR = resolve_model_dir(MODEL_NAME)

# ── 시작 LoRA 어댑터 : 공개된 파인튜닝 가중치에서 이어서 학습하거나 바로 추론 ──
#   이 노트북을 INIT_ADAPTER=None(처음부터) 기본 설정으로 학습한 어댑터 (검증 94.4%, TTA 2회 95.2%)
INIT_ADAPTER = "ssafyjinhyeok/KFC"   # None = 새 LoRA 로 원본 모델부터 학습. 로컬 폴더 경로도 가능

# ── 실험 설정 ──
SEED = 42

# 해상도 : Qwen3-VL 은 32x32 픽셀마다 비전 토큰 1개. 원본 비율을 유지한 채 "총 픽셀 수"를 이 범위로 맞춤
# (데이터 이미지 대부분이 720x960 ≈ 0.69MP 라서 1024x1024 예산이면 원본 크기 그대로 들어감. 더 키워도 거의 변화 없음)
TRAIN_MAX_PIXELS = 1024 * 1024             # 학습
INFER_MAX_PIXELS = 1024 * 1024             # 추론
MIN_PIXELS       = 128 * 32 * 32           # 이보다 작은 이미지는 이 크기까지 키움
MAX_SEQ_LEN      = 4096                    # 이미지 토큰 + 텍스트 토큰 상한 (잘리면 안 됨)

# 데이터
N_TRAIN         = None    # None = train 전체. 빠른 실험은 1000 등
VALID_SIZE      = 500     # train 에서 떼어 둘 검증 데이터 수 (정답이 확실한 train 으로 검증)
SPLIT_SEED      = 42      # 검증 분할용 시드. SEED 와 분리해서 모든 실험이 같은 검증 문항으로 비교·앙상블되게 함 (바꾸지 않기)
USE_DEV         = False   # dev.csv(5명 응답) 다수결 라벨을 학습에 추가할지. 라벨 노이즈가 커서 기본 False (아래 '데이터 준비' 설명)
DEV_MIN_VOTES   = 3       # 5명 중 3명 이상이 같은 답을 고른 문항만 사용
SHUFFLE_OPTIONS = True    # 학습 때 보기 순서를 매번 무작위로 섞기 (정답 글자도 함께 변경)

# LoRA
LORA_R, LORA_ALPHA = 16, 16
FINETUNE_VISION    = True    # 비전 인코더에도 LoRA (글자 인식 개선). VRAM 부족 시 False

# 학습
EPOCHS      = 1
LR          = 5e-5 if INIT_ADAPTER else 1e-4   # 이어서 학습은 더 작게. 처음부터도 원본이 이미 약 92% 라 너무 크면 능력을 망가뜨림
BATCH_SIZE  = 2
GRAD_ACCUM  = 4             # 유효 배치 = BATCH_SIZE x GRAD_ACCUM = 8
EVAL_STEPS  = 150           # optimizer step 마다 검증 정확도 측정 → 최고점 어댑터 저장 (1회 약 3분)
DO_TRAIN    = True          # False = 학습 없이 추론만 (같은 설정으로 학습한 어댑터가 있으면 그것, 없으면 INIT_ADAPTER 그대로)

# 프롬프트 : 학습·추론이 같은 프롬프트를 씀 ("a" 기본, "b" 글자 읽기 강조, "c" 한국어 시스템 프롬프트)
PROMPT_VARIANT = "a"
RUN_PROMPT_AB  = False      # True = 학습 전 모델로 세 프롬프트 점수 비교 (검증 300문항 x 3, 약 6분)

# 추론
N_TTA            = 2        # 보기 순서를 바꿔 여러 번 예측해 평균 (1 = 끔, 2 = 약 2배 시간, 4 = 약 4배)
INFER_BATCH_SIZE = 4

# 빠른 동작 확인 모드 : 전체 흐름이 에러 없이 도는지 몇 분 안에 확인
QUICK_RUN = os.environ.get("VQA_QUICK", "0") == "1"      # 직접 쓸 때는 True/False 로 바꿔도 됨
N_TEST = None
if QUICK_RUN:
    N_TRAIN, VALID_SIZE, EVAL_STEPS, N_TEST = 48, 24, 3, 40

# 시작 어댑터가 있으면 LoRA 구조(r, alpha, 비전 LoRA 여부)를 그 어댑터에 맞춤
INIT_ADAPTER_DIR = None
if INIT_ADAPTER:
    import json
    INIT_ADAPTER_DIR = resolve_model_dir(INIT_ADAPTER)
    _acfg = json.load(open(os.path.join(INIT_ADAPTER_DIR, "adapter_config.json"), encoding="utf-8"))
    LORA_R, LORA_ALPHA = _acfg["r"], _acfg["lora_alpha"]
    FINETUNE_VISION = "visual" in str(_acfg["target_modules"])
    if _acfg["base_model_name_or_path"].split("/")[-1] != MODEL_NAME.split("/")[-1]:
        print(f"⚠ 시작 어댑터의 베이스 모델({_acfg['base_model_name_or_path']})이 MODEL_NAME 과 다릅니다")
    # train.sample(500, random_state=42) 를 뺀 나머지로 학습됨 → 같은 검증 문항(또는 그 일부)을 써야 점수가 공정함
    assert SPLIT_SEED == 42 and VALID_SIZE <= 500, "시작 어댑터를 쓸 때는 SPLIT_SEED=42, VALID_SIZE<=500 을 유지하세요 (학습에 쓰인 문항이 검증에 섞임)"
    print(f"시작 어댑터 : {INIT_ADAPTER} → r={LORA_R}, alpha={LORA_ALPHA}, 비전 LoRA={FINETUNE_VISION}")

# 실험 이름 : 학습에 영향을 주는 설정값으로 자동 생성 (읽기 쉬운 앞부분 + 전체 학습 설정의 해시 6자리)
#   → 학습 설정을 바꾸면 어댑터·로그가 따로 저장됨 (앙상블에 활용)
#   → 추론 설정(N_TTA, INFER_MAX_PIXELS)은 INFER_TAG 로 확률·제출 파일 이름에만 붙음 (DO_TRAIN=False 로 추론만 다시 가능)
import hashlib
train_cfg = dict(model=MODEL_NAME, r=LORA_R, alpha=LORA_ALPHA, vision=FINETUNE_VISION, lr=LR, epochs=EPOCHS, seed=SEED,
                 batch=BATCH_SIZE, accum=GRAD_ACCUM, train_px=TRAIN_MAX_PIXELS, min_px=MIN_PIXELS, n_train=N_TRAIN,
                 valid=VALID_SIZE, split_seed=SPLIT_SEED, eval_steps=EVAL_STEPS, dev=USE_DEV, dev_votes=DEV_MIN_VOTES,
                 shuffle=SHUFFLE_OPTIONS, prompt=PROMPT_VARIANT, init=INIT_ADAPTER)
cfg_hash  = hashlib.md5(repr(sorted(train_cfg.items())).encode()).hexdigest()[:6]
MODEL_TAG = MODEL_NAME.split("/")[-1].split("-Instruct")[0].replace("-", "").lower()      # 예: qwen3vl4b
RUN_NAME  = f"{MODEL_TAG}_r{LORA_R}_lr{LR:g}_ep{EPOCHS}_s{SEED}_{cfg_hash}" + ("_quick" if QUICK_RUN else "")
INFER_TAG = f"tta{N_TTA}_px{int(INFER_MAX_PIXELS ** 0.5)}"
print("RUN_NAME:", RUN_NAME, "| INFER_TAG:", INFER_TAG)

ADAPTER_DIR = os.path.join(OUTPUT_DIR, f"{RUN_NAME}_best")
LOG_PATH    = os.path.join(OUTPUT_DIR, f"{RUN_NAME}_log.txt")   # 검증 정확도·loss 기록 (커널이 죽어도 남음)

def write_log(msg):
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(f"{time.strftime('%H:%M:%S')} {msg}\n")

# 사전 준비 폴더 확인
for name, path in [("모델", MODEL_DIR), ("데이터", DATA_DIR)] + ([("어댑터", INIT_ADAPTER_DIR)] if INIT_ADAPTER_DIR else []):
    state = "OK" if os.path.isdir(path) and os.listdir(path) else "없음 → 사전 준비 필요"
    print(f"{name:<4s} {state:<6s} {os.path.abspath(path)}")
print("QUICK_RUN:", QUICK_RUN)

# 라이브러리

⚠ `import unsloth` 는 **transformers / peft / trl 보다 먼저** 해야 Unsloth 최적화가 적용됩니다. (이미 transformers 를 import 한 커널이면 Restart 후 위에서부터 다시 실행)

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)

In [ ]:
import unsloth                                   # 반드시 가장 먼저
from unsloth import FastVisionModel
from unsloth.trainer import UnslothVisionDataCollator

import re, math, random, time
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset
from tqdm.auto import tqdm
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback
from peft import get_peft_model_state_dict, set_peft_model_state_dict

print(torch.__version__, "| CUDA:", torch.cuda.is_available())
assert torch.cuda.is_available(), "GPU 를 찾지 못했습니다. CUDA 버전 torch 가 설치된 커널인지 확인하세요."
print(torch.cuda.get_device_name(), "| bf16 지원:", torch.cuda.is_bf16_supported())

Image.MAX_IMAGE_PIXELS = None     # 큰 이미지 로드 허용
CHOICES = ["a", "b", "c", "d"]
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# 데이터 준비

- **검증 데이터** : train 에서 `VALID_SIZE`개를 떼어 냅니다. train 의 정답은 확실하므로 점수 비교 기준으로 씁니다.
- **dev.csv** : 정답 대신 5명의 응답(answer1~5, 일부 공란)이 있습니다. 다수결 라벨을 만들 수 있지만 **기본값은 학습에 쓰지 않습니다(`USE_DEV=False`).** 직접 확인한 근거는 다음과 같습니다.
  - 4명 이상 합의한 문항이 **하나도 없고**(최다 득표가 3표 또는 2표), 응답자 각각은 다수결과 60% 정도만 일치합니다.
  - 파인튜닝 전 모델이 train 검증 문항은 약 92% 맞히는데, dev 다수결과는 약 63%만 일치합니다. 틀린 문항 이미지를 직접 보면 다수결이 틀린 경우가 많습니다. (예: `dev_0348` 간판에 "홀랄라 치킨 1층"이 적혀 있는데 다수결은 "시골집")
  - test 는 질문 길이·모델 확신도 분포가 dev 가 아니라 train 과 거의 같습니다.

In [ ]:
train_all  = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test_df    = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
sample_sub = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))

# 검증 분리
valid_df = train_all.sample(n=VALID_SIZE, random_state=SPLIT_SEED).reset_index(drop=True)
train_df = train_all[~train_all["id"].isin(valid_df["id"])]
if N_TRAIN:
    train_df = train_df.sample(n=min(N_TRAIN, len(train_df)), random_state=SEED)

def majority_vote(row):
    """answer1~5 다수결 → (라벨, 득표수). 공동 1위면 라벨 None"""
    votes = [str(row[f"answer{k}"]).strip().lower() for k in range(1, 6) if pd.notna(row[f"answer{k}"])]
    votes = [v for v in votes if v in CHOICES]
    if not votes:
        return None, 0
    counts = {c: votes.count(c) for c in CHOICES}
    top = max(counts.values())
    winners = [c for c in CHOICES if counts[c] == top]
    return (winners[0] if len(winners) == 1 else None), top

dev_df = None
dev_path = os.path.join(DATA_DIR, "dev.csv")
if os.path.exists(dev_path):
    dev_df = pd.read_csv(dev_path)
    mv = dev_df.apply(majority_vote, axis=1)
    dev_df["answer"] = [m[0] for m in mv]
    dev_df["votes"]  = [m[1] for m in mv]
    dev_hi = dev_df[(dev_df["votes"] >= DEV_MIN_VOTES) & dev_df["answer"].notna()]
    print(f"dev {len(dev_df)}문항 중 {DEV_MIN_VOTES}표 이상 합의 {len(dev_hi)}문항")
    if USE_DEV and not QUICK_RUN:
        train_df = pd.concat([train_df, dev_hi[train_all.columns]], ignore_index=True)

train_df = train_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
if N_TEST:
    test_df = test_df.head(N_TEST)
print(f"train {len(train_df)} | valid {len(valid_df)} | test {len(test_df)}")
print("정답 분포(train):", train_df["answer"].value_counts().sort_index().to_dict())

# 모델 (Unsloth)

`FastVisionModel` 로 4bit 모델을 불러오고 LoRA 를 붙입니다. 디스크에서 로드하는 데 1~2분 걸립니다.

- `use_gradient_checkpointing="unsloth"` : Unsloth 방식 체크포인팅(활성값을 CPU 로 내려 VRAM 절약)
- `finetune_vision_layers` : 비전 인코더까지 LoRA 학습 (글자 인식 개선)
- `lora_dropout=0` : Unsloth 최적화 경로가 적용되는 값
- `INIT_ADAPTER` 가 있으면 그 LoRA 가중치(KFC)를 불러와 **시작점**으로 씁니다. 이어서 학습하거나 그대로 추론할 수 있습니다.

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - LoRA 구현 : LoraConfig()  →  Unsloth 에서는 FastVisionModel.get_peft_model()

In [ ]:
model, processor = FastVisionModel.from_pretrained(
    MODEL_DIR,
    load_in_4bit=True,                        # 4bit 양자화 (QLoRA)
    use_gradient_checkpointing="unsloth",
    max_seq_length=MAX_SEQ_LEN,
)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=FINETUNE_VISION,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    random_state=SEED,
)
model.print_trainable_parameters()
tokenizer = processor.tokenizer

from safetensors.torch import load_file

def load_adapter_weights(adapter_dir):
    """저장된 LoRA 가중치를 현재 모델에 덮어씀. 구조(r·비전 LoRA 여부)가 다르면 에러"""
    res = set_peft_model_state_dict(model, load_file(os.path.join(adapter_dir, "adapter_model.safetensors")))
    missing_lora = [k for k in res.missing_keys if "lora_" in k]
    assert not res.unexpected_keys and not missing_lora, \
        f"어댑터 구조가 현재 LoRA 설정과 다릅니다 (FINETUNE_VISION·LORA_R 확인): {(res.unexpected_keys or missing_lora)[:3]}"

if INIT_ADAPTER_DIR:
    load_adapter_weights(INIT_ADAPTER_DIR)
    print("시작 어댑터 로드:", INIT_ADAPTER)

# 학습 시작점 저장 : 학습 셀을 다시 실행해도 항상 이 상태에서 시작 (step 0 후보)
init_state = {k: v.detach().to("cpu", copy=True) for k, v in get_peft_model_state_dict(model).items()}
print(f"GPU 메모리 사용: {torch.cuda.memory_allocated() / 2**30:.1f} GB")

# 프롬프트 템플릿, 해상도

학습과 추론이 **같은 함수**로 프롬프트를 만들도록 해서 불일치를 막습니다.

해상도는 processor 의 이미지 크기 설정(`shortest_edge`=최소 픽셀 수, `longest_edge`=최대 픽셀 수)으로 조절합니다.
원본 비율은 유지되고 가로·세로 모두 32의 배수로 맞춰집니다. 원본이 예산보다 작으면 원본 크기 그대로(32 배수로 반올림) 씁니다.
이 데이터는 대부분 720×960(약 0.69MP)이라 1024×1024 예산이면 원본 그대로 들어가고, v3 의 384×384 예산은 픽셀 수를 약 1/5 로 줄였습니다.
(실측 zero-shot 정확도 : 512² 예산 58% → 768² 67% → 1024² 67%, dev 300문항 기준)

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - 프롬프트 템플릿 : convert_to_chatml(), formatting_prompts_func()

In [ ]:
SYSTEM_EN = ("You are a helpful visual question answering assistant. "
             "Answer using exactly one letter among a, b, c, or d. No explanation.")
SYSTEM_KO = ("당신은 이미지를 보고 질문에 답하는 도우미입니다. "
             "a, b, c, d 중 정답 한 글자만 답하고 설명은 하지 마세요.")
ANSWER_RULE = "정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요."
PROMPTS = {                                   # 변형 : (시스템 프롬프트, 질문 뒤에 붙는 지시문)
    "a": (SYSTEM_EN, ANSWER_RULE),                                                  # baseline v3 와 동일
    "b": (SYSTEM_EN, "이미지 속 글자를 한 글자씩 주의 깊게 읽고 답하세요. " + ANSWER_RULE),
    "c": (SYSTEM_KO, ANSWER_RULE),
}

def build_mc_prompt(question, a, b, c, d, variant=None):
    return (
        f"{question}\n"
        f"(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
        + PROMPTS[variant or PROMPT_VARIANT][1]
    )

def build_messages(img, question, options, answer=None, variant=None):
    """options : 화면에 보여줄 순서의 보기 4개. answer 가 있으면 학습용(assistant 턴 포함)"""
    variant = variant or PROMPT_VARIANT
    messages = [
        {"role": "system", "content": [{"type": "text", "text": PROMPTS[variant][0]}]},
        {"role": "user", "content": [
            {"type": "image", "image": img},
            {"type": "text", "text": build_mc_prompt(question, *options)},
        ]},
    ]
    if answer is not None:
        messages.append({"role": "assistant", "content": [{"type": "text", "text": answer}]})
    return messages

def set_image_budget(max_pixels, min_pixels=MIN_PIXELS):
    processor.image_processor.size = {"shortest_edge": int(min_pixels), "longest_edge": int(max_pixels)}

def load_image(rel_path):
    return Image.open(os.path.join(DATA_DIR, rel_path)).convert("RGB")

# 해상도 확인 : 실제로 모델에 들어가는 크기와 비전 토큰 수 (앞의 2장 + 가장 세로로 긴 이미지 1장)
set_image_budget(TRAIN_MAX_PIXELS)
sizes = {p: Image.open(os.path.join(DATA_DIR, p)).size for p in train_df["path"].head(300)}   # 헤더만 읽어서 빠름
tallest = max(sizes, key=lambda p: sizes[p][1] / sizes[p][0])
for p in list(train_df["path"].head(2)) + [tallest]:
    img = load_image(p)
    grid = processor.image_processor(images=[img], return_tensors="pt")["image_grid_thw"][0]
    h, w = int(grid[1]) * processor.image_processor.patch_size, int(grid[2]) * processor.image_processor.patch_size
    print(f"원본 {img.size[0]}x{img.size[1]} → 입력 {w}x{h}  (비전 토큰 {int(grid.prod()) // processor.image_processor.merge_size**2}개)")

# Custom Dataset, Collator (정답 응답 토큰만 학습)

- `VQATrainDataset` : 이미지는 필요할 때 디스크에서 읽습니다(메모리 절약). `SHUFFLE_OPTIONS=True` 면 매번 보기 순서를 섞고 정답 글자를 그에 맞게 바꿉니다.
- `UnslothVisionDataCollator(train_on_responses_only=True)` : `<|im_start|>assistant\n` 이후의 **응답 토큰만** loss 에 포함하고, 시스템·질문·이미지·패딩 토큰은 -100 으로 가립니다.
- 아래 셀 마지막에 **실제로 loss 에 들어가는 토큰**을 출력합니다. `정답 글자 + <|im_end|> + 줄바꿈` 3개만 남으면 정상입니다.
  (뒤의 두 토큰은 쉽게 맞히므로, 학습 로그의 loss 는 정답 글자 loss 의 약 1/3 크기로 보입니다)

#### 실습 참고 내용

    챕터 5-2 데이터 생성 및 파인튜닝
    - IntentDataset()

In [ ]:
class VQATrainDataset(Dataset):
    def __init__(self, df, shuffle_options=True, seed=SEED):
        self.df = df.reset_index(drop=True)
        self.shuffle_options = shuffle_options
        self.seed = seed
        self.n_seen = {}      # 같은 문항도 epoch 마다 다른 보기 순서가 되도록

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        options = [str(row[c]) for c in CHOICES]
        gold = CHOICES.index(str(row["answer"]).strip().lower())

        order = list(range(4))                               # order[j] = j번째 자리에 보여줄 원래 보기
        if self.shuffle_options:
            k = self.n_seen.get(i, 0); self.n_seen[i] = k + 1
            random.Random(self.seed * 1_000_003 + i * 7_919 + k).shuffle(order)
        shown = [options[j] for j in order]
        answer = CHOICES[order.index(gold)]

        img = load_image(row["path"])
        return {"messages": build_messages(img, str(row["question"]), shown, answer=answer)}

train_ds = VQATrainDataset(train_df, shuffle_options=SHUFFLE_OPTIONS)

collator = UnslothVisionDataCollator(
    model, processor,
    resize="max",                            # Unsloth 자체 리사이즈 끄기 → processor 픽셀 예산(set_image_budget)을 따름
    max_seq_length=MAX_SEQ_LEN,
    train_on_responses_only=True,            # 정답(assistant) 토큰에만 loss
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

# 라벨 마스킹 검증
set_image_budget(TRAIN_MAX_PIXELS)
check = collator([train_ds[0], train_ds[1]])
for ids, labels in zip(check["input_ids"], check["labels"]):
    kept = ids[labels != -100]
    print(f"시퀀스 길이 {len(ids):4d} | loss 대상 토큰 {len(kept)}개 → {tokenizer.decode(kept)!r}")
    assert 1 <= len(kept) <= 3 and tokenizer.decode(kept[:1]) in CHOICES, "정답 응답 외의 토큰이 loss 에 포함됨"

# 평가 함수 (a/b/c/d 확률 직접 비교 + 보기 순서 TTA)

`generate` 로 글자를 생성한 뒤 파싱하지 않고, 답을 쓰기 직전 위치에서 **`a`,`b`,`c`,`d` 토큰의 확률만** 비교합니다. 파싱 실패가 없고 forward 1회로 끝납니다.

**보기 순서 TTA** : 보기를 순환 이동한 프롬프트로도 예측해서 원래 자리로 되돌려 평균합니다. 모델이 특정 글자(예: a)를 선호하는 위치 편향을 줄입니다.

#### 실습 참고 내용

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 추론 : with torch.no_grad(), model.eval()

In [ ]:
CHOICE_IDS = [tokenizer.convert_tokens_to_ids(c) for c in CHOICES]
assert all(tokenizer.decode([t]) == c for t, c in zip(CHOICE_IDS, CHOICES))

@torch.inference_mode()
def predict_probs(df, max_pixels=INFER_MAX_PIXELS, n_tta=1, batch_size=INFER_BATCH_SIZE, desc="predict",
                  cache_path=None, cache_key="", variant=None):
    """df 의 각 문항에 대해 원래 보기 a~d 의 확률 (N, 4) 반환. variant : 프롬프트 변형 (None = PROMPT_VARIANT)
    cache_path 를 주면 중간 결과를 저장 → 끊겨도 같은 설정(cache_key·TTA·해상도·문항)이면 이어서 실행"""
    variant = variant or PROMPT_VARIANT
    shifts = [0, 2, 1, 3][:n_tta]              # 보기 순환 이동량 (2 = 순서를 반 바퀴 돌림)
    probs = np.zeros((len(df), 4))
    done = np.zeros(len(df), dtype=bool)
    key = (f"{cache_key}|prompt{variant}|tta{n_tta}|px{max_pixels}-{MIN_PIXELS}|n{len(df)}"
           f"|{df['id'].iloc[0]}|{df['id'].iloc[-1]}")
    if cache_path and os.path.exists(cache_path):
        with np.load(cache_path) as saved:     # with : 파일을 바로 닫음 (Windows 에서 나중에 삭제 가능하도록)
            ok = "key" in saved.files and str(saved["key"]) == key
            if ok:
                probs, done = saved["probs"], saved["done"]
        print(f"[{desc}] " + (f"저장된 진행분 {done.sum()}/{len(df)} 이어서 실행" if ok else "다른 모델/설정의 캐시라서 무시하고 처음부터 실행"))

    old_size, old_side, was_training = dict(processor.image_processor.size), tokenizer.padding_side, model.training
    set_image_budget(max_pixels)
    tokenizer.padding_side = "left"            # 배치 추론 : 왼쪽 패딩이어야 마지막 위치가 답 자리
    model.eval()
    try:
        todo = np.where(~done)[0]
        for n_batch, start in enumerate(tqdm(range(0, len(todo), batch_size), desc=desc)):
            idx = todo[start:start + batch_size]
            rows = [df.iloc[i] for i in idx]
            imgs = [load_image(r["path"]) for r in rows]
            acc = np.zeros((len(idx), 4))
            for s in shifts:
                order = [(j + s) % 4 for j in range(4)]    # order[j] = j번째 자리에 보여줄 원래 보기
                texts = [
                    processor.apply_chat_template(
                        build_messages(img, str(r["question"]), [str(r[CHOICES[o]]) for o in order], variant=variant),
                        tokenize=False, add_generation_prompt=True)
                    for r, img in zip(rows, imgs)
                ]
                enc = processor(text=texts, images=imgs, padding=True, return_tensors="pt").to(model.device)
                with torch.autocast("cuda", dtype=torch.bfloat16):
                    logits = model(**enc, logits_to_keep=1).logits[:, -1, CHOICE_IDS]
                p = torch.softmax(logits.float(), dim=-1).cpu().numpy()
                acc[:, order] += p                         # j번째 자리의 확률 → 원래 보기 order[j] 자리로 누적
            probs[idx] = acc / len(shifts)
            done[idx] = True
            if cache_path and (n_batch % 50 == 49 or start + batch_size >= len(todo)):
                np.savez(cache_path, probs=probs, done=done, key=np.array(key))
    finally:                                   # 중간에 멈춰도 학습용 설정으로 되돌림
        processor.image_processor.size = old_size
        tokenizer.padding_side = old_side
        if was_training:
            model.train()
    return probs

def evaluate(df, probs, name="valid", show_wrong=5):
    gold = df["answer"].str.strip().str.lower().map(CHOICES.index).values
    pred = probs.argmax(1)
    correct = pred == gold
    numeric = df[CHOICES].astype(str).apply(lambda r: all(re.fullmatch(r"[\d\s,.\-:/%원₩$+()]+", x) for x in r), axis=1).values
    print(f"[{name}] 정확도 {correct.mean():.4f} ({correct.sum()}/{len(df)})"
          f" | 숫자형 보기 {correct[numeric].mean():.3f} (n={numeric.sum()})"
          f" | 텍스트형 보기 {correct[~numeric].mean():.3f} (n={(~numeric).sum()})")
    print("   예측 분포:", dict(zip(CHOICES, np.bincount(pred, minlength=4).tolist())))
    if show_wrong:
        wrong = df[~correct].head(show_wrong)
        for (_, r), pr in zip(wrong.iterrows(), pred[~correct][:show_wrong]):
            print(f"   ✗ {r['id']} | {r['question'][:40]} | 예측 {CHOICES[pr]}={str(r[CHOICES[pr]])[:15]} / 정답 {r['answer']}={str(r[r['answer']])[:15]}")
    return correct.mean()

# 시작 모델 점수 (step 0 기준선)

학습 전 모델의 검증 점수입니다. 파인튜닝 효과를 이 값과 비교합니다.
- `INIT_ADAPTER = None` : LoRA 가 0 으로 초기화되어 있어서 원본 모델(zero-shot) 점수 (실측 91.6%)
- `INIT_ADAPTER = "ssafyjinhyeok/KFC"` : 어댑터의 점수 (실측 94.4%)

먼저 배치 추론(왼쪽 패딩)이 한 장씩 추론한 결과와 같은 답을 내는지 확인합니다.

In [ ]:
# 배치 추론 검증 : 한 장씩(batch 1) vs 배치(INFER_BATCH_SIZE) 예측이 같은 답인지 (bf16·패딩 때문에 확률은 약간 다를 수 있음)
chk = valid_df.head(8)
p1, pb = predict_probs(chk, batch_size=1, desc="batch=1"), predict_probs(chk, desc=f"batch={INFER_BATCH_SIZE}")
print(f"배치 vs 단일 : 답 일치 {(p1.argmax(1) == pb.argmax(1)).mean():.0%}, 확률 최대 차이 {np.abs(p1 - pb).max():.3f}")

t0 = time.time()
start_probs = predict_probs(valid_df, n_tta=1, desc="step 0 valid")
start_acc = evaluate(valid_df, start_probs, name="시작 모델 valid")
sec_per_sample = (time.time() - t0) / len(valid_df)
write_log(f"step 0 ({'INIT_ADAPTER' if INIT_ADAPTER else 'zero-shot'}) valid_acc {start_acc:.4f}")
print(f"추론 속도 {sec_per_sample:.2f}초/문항 (TTA 없음). test 추론 시간은 '최종 검증' 셀에서 TTA 포함 속도로 다시 추정합니다")

# (선택) 프롬프트 A/B 테스트

`RUN_PROMPT_AB = True` 이면 세 가지 프롬프트의 점수를 같은 검증 문항으로 비교합니다. (`INIT_ADAPTER = None` 인 원본 모델에서 비교하세요. 프롬프트 "a" 로 학습되어 "a" 가 유리합니다)
가장 좋은 변형을 설정 셀의 `PROMPT_VARIANT` 에 넣고 **모델 셀부터 다시 실행**하세요. (학습·추론이 같은 프롬프트를 쓰도록 자동으로 맞춰짐)

In [ ]:
if RUN_PROMPT_AB or QUICK_RUN:                 # QUICK_RUN 에서는 코드가 도는지만 확인
    ab_df = valid_df.head(300)
    for v in PROMPTS:
        evaluate(ab_df, predict_probs(ab_df, n_tta=1, desc=f"prompt {v}", variant=v), name=f"prompt {v}", show_wrong=0)
else:
    print("건너뜀 (RUN_PROMPT_AB = False)")

# fine-tuning (Unsloth + TRL SFTTrainer)

- `EVAL_STEPS` 마다 검증 정확도를 재고, **최고 정확도일 때의 어댑터를 저장**합니다. 학습이 끝나면(또는 ■ 버튼으로 중단하면) 최고점 어댑터로 되돌립니다.
- 학습 전 시작 모델(원본)도 step 0 후보에 들어갑니다. 파인튜닝이 오히려 점수를 떨어뜨리면 시작 모델로 돌아갑니다.
- `INIT_ADAPTER` 가 있으면 그 가중치에서 **이어서 학습**합니다. (기본 LR 5e-5, 1 epoch)
- 학습 시간은 진행 막대의 남은 시간을 참고하세요. (RTX 5060 Ti 실측 : 4B·1024px 에서 약 1.2초/문항 → train 6.2천 문항 약 2시간 + 검증 평가 1회 약 3분)

#### 실습 참고 내용

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6: 모델 학습을 위한 반복문

In [ ]:
class BestValidCallback(TrainerCallback):
    """EVAL_STEPS 마다 검증 정확도 측정 → 최고점이면 LoRA 가중치 저장.
    학습 전 상태(시작 모델, step 0 점수)를 step 0 후보로 먼저 저장해 둠"""
    def __init__(self, model, start_acc):
        self.history = [(0, start_acc)]
        self._save_best(model, start_acc, 0)

    def _save_best(self, model, acc, step):
        self.best_acc, self.best_step = acc, step
        self.best_state = {k: v.detach().to("cpu", copy=True) for k, v in get_peft_model_state_dict(model).items()}
        model.save_pretrained(ADAPTER_DIR)

    def _eval(self, state, model):
        probs = predict_probs(valid_df, n_tta=1, desc=f"valid@{state.global_step}")
        acc = evaluate(valid_df, probs, name=f"step {state.global_step}", show_wrong=0)
        self.history.append((state.global_step, acc))
        write_log(f"step {state.global_step}/{state.max_steps} valid_acc {acc:.4f}")
        if acc > self.best_acc:
            self._save_best(model, acc, state.global_step)
            print(f"   ★ best 갱신 → {ADAPTER_DIR}")

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            write_log(f"step {state.global_step}/{state.max_steps} loss {logs['loss']:.4f} lr {logs.get('learning_rate', 0):.2e}")

    def on_step_end(self, args, state, control, model=None, **kwargs):
        if state.global_step % EVAL_STEPS == 0 or state.global_step == state.max_steps:
            self._eval(state, model)

if DO_TRAIN:
    set_peft_model_state_dict(model, init_state)   # 항상 시작점에서 출발 (이 셀을 다시 실행해도 step 0 점수와 일치)
    set_image_budget(TRAIN_MAX_PIXELS)
    FastVisionModel.for_training(model)
    best_cb = BestValidCallback(model, start_acc)

    trainer = SFTTrainer(
        model=model,
        processing_class=tokenizer,
        data_collator=collator,
        train_dataset=train_ds,
        callbacks=[best_cb],
        args=SFTConfig(
            output_dir=os.path.join(OUTPUT_DIR, "trainer"),
            per_device_train_batch_size=BATCH_SIZE,
            gradient_accumulation_steps=GRAD_ACCUM,
            num_train_epochs=EPOCHS,
            learning_rate=LR,
            lr_scheduler_type="cosine",
            warmup_ratio=0.05,
            weight_decay=0.01,
            optim="adamw_8bit",
            max_grad_norm=1.0,
            bf16=True,
            logging_steps=10,
            save_strategy="no",             # 저장은 BestValidCallback 이 담당
            report_to="none",
            seed=SEED,
            # --- 비전 데이터용 필수 설정 (Unsloth 권장) ---
            remove_unused_columns=False,
            dataset_text_field="",
            dataset_kwargs={"skip_prepare_dataset": True},
            max_length=MAX_SEQ_LEN,
            dataloader_num_workers=0,        # Windows 노트북 호환
        ),
    )
    t0 = time.time()
    try:
        train_result = trainer.train()
        print(f"학습 시간 {(time.time() - t0) / 60:.1f}분 | 평균 train loss {train_result.training_loss:.4f}")
    except KeyboardInterrupt:
        print("⚠ 학습을 중단했습니다 → 지금까지의 best 어댑터로 복원합니다")
    train_min = (time.time() - t0) / 60
    write_log(f"train end {train_min:.1f} min, best step {best_cb.best_step} valid_acc {best_cb.best_acc:.4f}")
    print("검증 정확도 기록 (step, 정확도):", [(s, round(float(a), 4)) for s, a in best_cb.history])

    # 최고점 어댑터로 되돌리기
    set_peft_model_state_dict(model, best_cb.best_state)
    print(f"best step {best_cb.best_step} (검증 {best_cb.best_acc:.4f}) 어댑터로 복원")
    if best_cb.best_step == 0:
        print("⚠ 어떤 checkpoint 도 시작 모델보다 좋지 않아 시작 모델을 그대로 사용합니다."
              + (" (QUICK_RUN 은 검증 문항이 적어 정상일 수 있음)" if QUICK_RUN else " LR·데이터를 점검하세요."))
else:
    # 학습 없이 추론 : 같은 설정으로 학습해 둔 어댑터가 있으면 그것, 없으면 시작 어댑터(INIT_ADAPTER) 그대로
    train_min = 0.0
    if os.path.exists(os.path.join(ADAPTER_DIR, "adapter_model.safetensors")):
        import json
        cfg = json.load(open(os.path.join(ADAPTER_DIR, "adapter_config.json"), encoding="utf-8"))
        assert (cfg["r"], cfg["lora_alpha"]) == (LORA_R, LORA_ALPHA), \
            f"저장된 어댑터(r={cfg['r']}, alpha={cfg['lora_alpha']})와 현재 LORA_R/LORA_ALPHA 설정이 다릅니다"
        load_adapter_weights(ADAPTER_DIR)
        print("저장된 어댑터 로드:", ADAPTER_DIR)
    elif INIT_ADAPTER_DIR:
        set_peft_model_state_dict(model, init_state)
        model.save_pretrained(ADAPTER_DIR)          # 이후 셀(캐시·재실행)에서 같은 경로로 쓰도록 저장
        print("시작 어댑터로 바로 추론:", INIT_ADAPTER)
    else:
        raise FileNotFoundError(f"{ADAPTER_DIR} 에 학습된 어댑터가 없습니다. DO_TRAIN=True 로 먼저 학습하거나 INIT_ADAPTER 를 지정하세요")

_ = processor.save_pretrained(ADAPTER_DIR)

# 최종 검증

최고점 어댑터로 검증 데이터를 TTA 와 함께 다시 평가합니다.

- 검증 확률 `output/probs_valid_{RUN_NAME}_{INFER_TAG}.npy` 저장 → 앙상블 조합을 검증 점수로 고를 때 사용
- 실험 결과 한 줄을 `output/experiments.csv` 에 누적 → 실험끼리 비교
- 모델이 **높은 확신(>0.95)으로 라벨과 다르게 답한 문항**을 `output/label_suspects_{RUN_NAME}.csv` 로 저장 → 라벨 오류 후보를 눈으로 확인

In [ ]:
FastVisionModel.for_inference(model)
t0 = time.time()
valid_probs = predict_probs(valid_df, n_tta=N_TTA, desc="valid (TTA)")
print(f"TTA{N_TTA} 추론 {(time.time() - t0) / len(valid_df):.2f}초/문항 → test {len(test_df)}문항 예상 약 {(time.time() - t0) / len(valid_df) * len(test_df) / 60:.0f}분")
final_acc = evaluate(valid_df, valid_probs, name=f"valid TTA{N_TTA}")
write_log(f"final valid_acc TTA{N_TTA} {final_acc:.4f}")
if DO_TRAIN:
    print(f"시작 모델 {start_acc:.4f} → 파인튜닝(best step {best_cb.best_step}, TTA 없음) {best_cb.best_acc:.4f}"
          f" → + TTA{N_TTA} {final_acc:.4f}")
else:
    print(f"시작 모델(TTA 없음) {start_acc:.4f} | 사용한 어댑터 + TTA{N_TTA} {final_acc:.4f}")
print(f"※ 검증 {len(valid_df)}문항 기준 표준오차 약 ±{np.sqrt(final_acc * (1 - final_acc) / len(valid_df)) * 100:.1f}%p"
      " → 이보다 작은 차이는 우연일 수 있음")
np.save(os.path.join(OUTPUT_DIR, f"probs_valid_{RUN_NAME}_{INFER_TAG}.npy"), valid_probs)

# 실험 기록 누적
if not QUICK_RUN:
    exp_path = os.path.join(OUTPUT_DIR, "experiments.csv")
    row = {"time": time.strftime("%Y-%m-%d %H:%M"), "run": RUN_NAME, "infer": INFER_TAG, **train_cfg,
           "start": round(start_acc, 4), "best_step": best_cb.best_step if DO_TRAIN else None,
           "best_valid": round(best_cb.best_acc, 4) if DO_TRAIN else None, "final_valid": round(final_acc, 4),
           "train_min": round(train_min, 1)}
    pd.DataFrame([row]).to_csv(exp_path, mode="a", header=not os.path.exists(exp_path), index=False, encoding="utf-8-sig")
    print("실험 기록 →", exp_path)

# 라벨 오류 후보 : 모델이 높은 확신으로 라벨과 다르게 답한 검증 문항
pred, pmax = valid_probs.argmax(1), valid_probs.max(1)
gold = valid_df["answer"].str.strip().str.lower().map(CHOICES.index).values
suspect = (pmax > 0.95) & (pred != gold)
sus_df = valid_df.loc[suspect, ["id", "path", "question", *CHOICES, "answer"]].assign(
    pred=[CHOICES[i] for i in pred[suspect]], prob=pmax[suspect].round(3))
sus_df.to_csv(os.path.join(OUTPUT_DIR, f"label_suspects_{RUN_NAME}.csv"), index=False, encoding="utf-8-sig")
print(f"라벨 오류 후보 {len(sus_df)}문항 → output/label_suspects_{RUN_NAME}.csv (이미지를 열어 직접 확인해 보세요)")

# inference (test) + 제출 파일

- 제출 파일 : **`output/submission.csv`** (실험별 사본 `output/submission_{RUN_NAME}_{INFER_TAG}.csv`)
- 문항별 a~d 확률을 `output/probs_test_{RUN_NAME}_{INFER_TAG}.npy` 로 저장합니다. 여러 실험의 확률을 평균하면 앙상블이 됩니다.
- 중간에 끊겨도 같은 셀을 다시 실행하면 이어서 예측합니다. (같은 어댑터·TTA·해상도일 때만. 다 끝나면 이어하기 캐시는 삭제)

#### 실습 참고 내용

    챕터4-1 RAG 기반 Customer Service AI 에이전트 개발
    - 데이터 파서 : 문자열 파싱 대신 토큰 확률로 선택지 결정

In [ ]:
cache_path = os.path.join(OUTPUT_DIR, f"cache_test_{RUN_NAME}.npz")
adapter_id = os.path.getmtime(os.path.join(ADAPTER_DIR, "adapter_model.safetensors"))   # 어댑터가 바뀌면 캐시 무효
test_probs = predict_probs(test_df, n_tta=N_TTA, desc=f"test (TTA{N_TTA})",
                           cache_path=cache_path, cache_key=f"{RUN_NAME}|{adapter_id}")
np.save(os.path.join(OUTPUT_DIR, f"probs_test_{RUN_NAME}_{INFER_TAG}.npy"), test_probs)
if os.path.exists(cache_path):
    os.remove(cache_path)                      # 끝까지 예측했으면 이어하기용 캐시는 삭제

submission = pd.DataFrame({"id": test_df["id"].values, "answer": [CHOICES[i] for i in test_probs.argmax(1)]})

# 제출 전 점검
assert submission["answer"].isin(CHOICES).all()
if N_TEST is None:
    assert len(submission) == len(sample_sub) and (submission["id"].values == sample_sub["id"].values).all(), "id 순서/개수가 sample_submission 과 다름"
dist = submission["answer"].value_counts(normalize=True).sort_index()
print("예측 분포:", dist.round(3).to_dict())
assert dist.max() < 0.5, "한 글자로 예측이 쏠림 → 모델/프롬프트 점검 필요"

SUBMISSION_PATH = os.path.join(OUTPUT_DIR, f"submission_{RUN_NAME}_{INFER_TAG}.csv")
submission.to_csv(SUBMISSION_PATH, index=False)
if N_TEST is None:                             # QUICK_RUN(일부 문항)은 제출용 파일을 덮어쓰지 않음
    submission.to_csv(os.path.join(OUTPUT_DIR, "submission.csv"), index=False)
print("Saved", SUBMISSION_PATH, "" if N_TEST else "+ output/submission.csv")
write_log(f"saved {SUBMISSION_PATH}")
submission.head()

# 다음 단계 (점수를 더 올리려면)

학습 설정을 바꾸면 `RUN_NAME`(설정값 + 해시)이 바뀌어 어댑터·로그·확률·제출 파일이 실험별로 따로 저장됩니다.
추론 설정(`N_TTA`, `INFER_MAX_PIXELS`)만 바꿀 때는 `DO_TRAIN = False` 로 학습을 건너뛰면 저장된 어댑터로 추론만 다시 하고, 파일 이름의 `INFER_TAG` 부분만 달라집니다.

| 방법 | 설정 | 비고 |
|---|---|---|
| 더 큰 모델 | `MODEL_NAME` 을 8B 로 | 사전 다운로드 필요, 학습·추론 약 2배, 16GB 에 `BATCH_SIZE=1, GRAD_ACCUM=8` 권장 |
| 앙상블 | 맨 아래 셀 : 실험별 확률 평균 → 검증 점수 확인 후 제출 | `SEED`·`LR`·모델을 바꾼 실험끼리 섞을수록 효과 큼. 같은 어댑터의 TTA 변형은 1개만 사용 |
| 프롬프트 | `RUN_PROMPT_AB = True` 로 비교 → `PROMPT_VARIANT` 변경 | 학습·추론 프롬프트가 자동으로 같이 바뀜 |
| epoch 2 | `EPOCHS = 2` | 학습 시간 2배. 검증 정확도가 꺾이면 best 어댑터가 자동 선택됨 |
| TTA | `N_TTA = 4`, `DO_TRAIN = False` | 실측 : TTA 2회가 파인튜닝 모델에서 +0.8%p(검증 500), zero-shot 에서 +3.7%p(dev 300). 4회는 zero-shot 에서 추가 이득 없음 → 기본 2회 |
| 작은 글자 확대 | `MIN_PIXELS` 를 키워 업스케일 (학습·추론 모두) | 비전 토큰·시간 크게 증가, 효과 미검증 |

※ `TRAIN_MAX_PIXELS` 를 1024² 보다 키워도 대부분 이미지(720×960)는 그대로라 거의 변화가 없습니다.

**OOM 이 나면** : `BATCH_SIZE=1, GRAD_ACCUM=8` → `FINETUNE_VISION=False` → `TRAIN_MAX_PIXELS` 축소(768² 까지는 zero-shot 점수 동일) → `LORA_R=8` 순서로 줄이세요.

In [ ]:
# 앙상블 : 저장해 둔 실험들의 확률 평균. 실험(RUN_NAME)마다 가장 최근 추론 결과 1개만 사용
import glob
latest = {}
for f in sorted(glob.glob(os.path.join(OUTPUT_DIR, "probs_test_*.npy")), key=os.path.getmtime):
    if "_quick" in f:
        continue
    run = os.path.basename(f)[len("probs_test_"):].rsplit("_tta", 1)[0]
    valid_f = f.replace("probs_test_", "probs_valid_")
    if os.path.exists(valid_f):                # 검증 확률이 있는 실험만 (같은 SPLIT_SEED·VALID_SIZE 여야 비교 가능)
        latest[run] = (f, valid_f)

if len(latest) >= 2 and not QUICK_RUN:
    runs = [r for r in latest if np.load(latest[r][1]).shape == (len(valid_df), 4)]
    for r in runs:
        evaluate(valid_df, np.load(latest[r][1]), name=r, show_wrong=0)
    ens_valid = np.mean([np.load(latest[r][1]) for r in runs], axis=0)
    evaluate(valid_df, ens_valid, name=f"앙상블 {len(runs)}개", show_wrong=0)
    ens = np.mean([np.load(latest[r][0]) for r in runs], axis=0)
    ens_sub = pd.DataFrame({"id": test_df["id"].values, "answer": [CHOICES[i] for i in ens.argmax(1)]})
    ens_sub.to_csv(os.path.join(OUTPUT_DIR, "submission_ensemble.csv"), index=False)
    print("Saved output/submission_ensemble.csv  (앙상블 검증 점수가 단일 실험보다 높을 때만 제출하세요)")
else:
    print(f"앙상블할 실험이 {len(latest)}개 → 2개 이상 필요 (SEED·LR·모델을 바꿔 한 번 더 실행)")